> **Outputs intentionally cleared.** Run the canonical pipeline before executing this notebook. The authoritative current numbers and hashes are generated in `docs/findings.md` by `thesis report findings`; notebook cells are exploratory views, not committed evidence.


# Data Quality Review — Silver Layer

This notebook explores the canonical silver and gold Parquet tables produced by `python -m thesis pipeline run`. The authoritative DQ evidence is generated by `python -m thesis dq run` in `data/dq/*.dq.{json,md}`.

**Medallion layout on disk:**

```
data/
 ├── raw/                    # bronze — untouched OCDS JSONL
 ├── silver/
 │   ├── tenders.parquet     # 1 row per tender
 │   └── awards.parquet      # 1 row per (award, supplier), linked to tenders
 ├── gold/
 │   ├── tender_features.parquet  # immutable features + rules
 │   └── tender_summary.parquet   # ML-scored table
 ├── models/                 # benchmark, JSON and joblib artifacts
 └── dq/                     # canonical *.dq.json / *.dq.md evidence
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from thesis.domain.features import (normalise_supplier_business_id, supplier_key,
                                      supplier_source_identity_key)
from thesis.domain.procedures import PROCEDURE_CODE_MAP

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 4.5)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

ROOT = Path('..')
SILVER = ROOT / 'data' / 'silver'
GOLD = ROOT / 'data' / 'gold'
DQ = ROOT / 'data' / 'dq'

tenders = pd.read_parquet(SILVER / 'tenders.parquet')
awards = pd.read_parquet(SILVER / 'awards.parquet')
print(f'tenders: {len(tenders):,} rows, {tenders.shape[1]} cols')
print(f'awards:  {len(awards):,} rows, {awards.shape[1]} cols')

## 1. Dataset overview

In [ ]:
overview = pd.Series({
    'tenders': len(tenders),
    'awards': len(awards),
    'tenders_with_awards': int((tenders.num_awards >= 1).sum()),
    'tenders_without_awards': int((tenders.num_awards == 0).sum()),
    'date_min': tenders.release_date.min(),
    'date_max': tenders.release_date.max(),
    'procuring_entities': tenders.procuring_entity_id.nunique(),
    'distinct_supplier_business_ids': awards.supplier_business_id.nunique(),
    'total_estimated_eur_bn': round(tenders.estimated_value_eur.sum() / 1e9, 2),
    'total_awarded_eur_bn':   round(awards.awarded_value_eur.sum() / 1e9, 2),
})
overview.to_frame('value')

### Award-to-lot title linkage

Awards publish a free-text lot title, not a shared lot identifier. The Silver status below reports only exact-title linkage; ambiguous, unmatched, and missing titles are not silently joined. All award-value ratios remain tender-level diagnostics.

In [ ]:
awards['award_lot_match_status'].value_counts(dropna=False).rename('award rows').to_frame()

## 2. Silver-column coverage

Coverage is recomputed directly from the current canonical silver tender table. Raw nested-path profiling is a separate source audit and is not reused as live pipeline evidence.

In [ ]:
coverage = tenders.notna().mean().mul(100).sort_values()
plt.figure(figsize=(10, 10))
colors = coverage.apply(lambda x: 'steelblue' if x >= 90 else ('orange' if x >= 50 else 'crimson'))
plt.barh(coverage.index, coverage.values, color=colors)
plt.axvline(100, color='black', linestyle=':', linewidth=0.8)
plt.xlabel('non-null coverage (% of silver tender rows)')
plt.title(f'Silver tender-column coverage — {len(tenders):,} records')
plt.tight_layout();

## 3. Missingness in the silver tender table

Missingness is recomputed from the current table. The canonical column is `method`; null and blank values remain an explicit unknown bucket in the dashboard.

In [ ]:
missing = tenders.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    ax = missing.plot(kind='barh', color='indianred', figsize=(9, 4))
    ax.set_xlabel('% missing')
    ax.set_title('Columns with any missing values (silver.tenders)')
    plt.tight_layout()
else:
    print('No missing values in silver.tenders.')

## 4. The `end_before_start` anomaly — decomposed by method

Historical diagnostic example: decomposing `endDate < startDate` by source procedure detail can reveal differing date semantics. Recompute the current rates from the regenerated silver/gold artifacts; the coarse `open` field is not treated as proof of a standard-open or EU-equivalent legal regime.

In [ ]:
bm = tenders.groupby('method')['end_before_start'].agg(['sum', 'count'])
bm['pct'] = (bm['sum'] / bm['count'] * 100).round(2)
bm = bm.sort_values('count', ascending=False)
display(bm)

ax = bm['pct'].plot(kind='bar', color='darkred', figsize=(9, 4))
ax.set_title('`end_before_start` rate by procurement method')
ax.set_ylabel('% of tenders flagged')
for i, v in enumerate(bm['pct']):
    ax.text(i, v + 1, f'{v}%', ha='center')
plt.tight_layout();

## 5. Award timing — days from tender end to award date

This cell describes recorded date ordering only. Negative or very long intervals require review of source semantics, corrections and documents; they do not by themselves establish backdating, delay, governance quality or process cause.

In [ ]:
timing = (awards[['ocid', 'award_date']].dropna()
          .merge(tenders[['ocid', 'end_date', 'method']], on='ocid', how='left'))
timing['days_to_award'] = (timing['award_date'] - timing['end_date']).dt.total_seconds() / 86400
t = timing['days_to_award'].dropna()
print('Summary:')
print(t.describe(percentiles=[.05, .5, .95]).round(1))
print(f"\nNegative (award before end): {(t < 0).sum():,}")
print(f">180 days: {(t > 180).sum():,}")

clipped = t.clip(-30, 365)
plt.figure(figsize=(10, 4))
plt.hist(clipped, bins=80, color='teal', edgecolor='black', linewidth=0.3)
plt.axvline(0, color='red', linestyle='--', label='tender end')
plt.axvline(t.median(), color='black', linestyle=':', label=f'median={t.median():.0f}d')
plt.legend()
plt.xlabel('days from tender end to award (clipped -30..365)')
plt.title('Award-timing distribution')
plt.tight_layout();

## 6. Awarded value vs. estimated — a comparability diagnostic

The ratio `awarded / estimated` can highlight records for data-quality review, but tender-level awards and lot-level estimates are not always directly comparable. An OCDS `award` is a recorded award decision; this extract does not establish a signed, executed or paid contract:

* ratio ≈ 1: recorded totals are close
* ratio ≪ 1: possible partial/lot award, missing value, or incomparable aggregation
* ratio > 1: possible lot/aggregation mismatch or field/unit comparability issue


In [ ]:
rat = (awards[['awarded_value_eur', 'estimated_value_eur', 'category', 'method']]
       .query('awarded_value_eur > 0 and estimated_value_eur > 0')
       .assign(ratio=lambda d: d.awarded_value_eur / d.estimated_value_eur))

print('ratio percentiles:')
print(rat.ratio.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# log-scaled histogram — a lot of mass below 1
clip = rat.ratio.clip(1e-3, 3)
axes[0].hist(np.log10(clip), bins=60, color='steelblue', edgecolor='black', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', label='ratio = 1')
axes[0].set_xlabel('log10(awarded / estimated)')
axes[0].set_title('Award ratio distribution')
axes[0].legend()

sns.boxplot(data=rat.assign(log_ratio=np.log10(rat.ratio.clip(1e-3, 3))),
            x='category', y='log_ratio', ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Award ratio (log10) by category')
plt.tight_layout();

## 7. Descriptive supplier concentration and identity coverage

Three sensitivity views:
* **Descriptive HHI** under business-ID-only, ID/name/source-party, and ID/source-party/name keys, always paired with row/value coverage. Source party IDs are publication identities, not verified legal IDs. Broad procurement categories are not relevant markets.
* **Lorenz curve**: cumulative share of identified awarded value vs. cumulative share of resolved supplier keys.

In [ ]:
work = awards.loc[awards.awarded_value_eur.notna()].copy()
work['id_key'] = work.supplier_business_id.map(normalise_supplier_business_id)
work['name_first_key'] = supplier_key(work)
work['source_party_first_key'] = supplier_source_identity_key(work)

def concentration_row(label, key):
    identified = work.dropna(subset=[key])
    values = identified.groupby(key).awarded_value_eur.sum()
    shares = values / values.sum()
    return {
        'key': label, 'supplier_groups': len(values),
        'row_coverage_pct': 100 * len(identified) / len(work),
        'value_coverage_pct': 100 * identified.awarded_value_eur.sum() / work.awarded_value_eur.sum(),
        'descriptive_hhi': (shares.pow(2).sum() * 10000),
        'top_10_identified_value_pct': (shares.nlargest(10).sum() * 100),
    }

display(pd.DataFrame([concentration_row('business ID only', 'id_key'),
                      concentration_row('ID / name / source party', 'name_first_key'),
                      concentration_row('ID / source party / name', 'source_party_first_key')])
        .round(2))

a = (work.dropna(subset=['source_party_first_key'])
         .groupby('source_party_first_key').awarded_value_eur.sum().sort_values())
cum_supp = np.arange(1, len(a) + 1) / len(a)
cum_val = np.cumsum(a.values) / a.sum()
plt.figure(figsize=(6, 6))
plt.plot(cum_supp, cum_val, color='darkblue', label='Lorenz curve')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='perfect equality')
plt.xlabel('cumulative share of suppliers')
plt.ylabel('cumulative share of awarded value')
plt.title('Supplier-side Lorenz curve')
plt.legend()
plt.tight_layout();

## 8. Source-procedure diagnostic matrix

In [ ]:
observed = tenders.bidder_data_observed.fillna(False)
method_profile = (tenders.assign(
        procedure_subtype=tenders.procedure_code.astype('string').map(PROCEDURE_CODE_MAP),
        was_awarded=tenders.num_awards.ge(1),
        bidder_data_observed=observed,
        single_bidder=tenders.num_bidders.eq(1).where(observed),
        zero_bidder=tenders.num_bidders.eq(0).where(observed))
    .groupby('procedure_subtype', dropna=False).agg(
        tenders=('ocid', 'size'), awarded_pct=('was_awarded', 'mean'),
        bidder_data_coverage_pct=('bidder_data_observed', 'mean'),
        single_bidder_pct=('single_bidder', 'mean'),
        zero_bidder_pct=('zero_bidder', 'mean'),
        end_before_start_pct=('end_before_start', 'mean'),
        median_estimated_eur=('estimated_value_eur', 'median')))
for c in ['awarded_pct', 'bidder_data_coverage_pct', 'single_bidder_pct',
          'zero_bidder_pct', 'end_before_start_pct']:
    method_profile[c] = 100 * method_profile[c]
display(method_profile.round(2))

plot = method_profile[['bidder_data_coverage_pct', 'single_bidder_pct',
                       'zero_bidder_pct', 'end_before_start_pct']]
ax = plot.plot(kind='bar', figsize=(10, 4.5), colormap='RdYlBu_r')
ax.set_title('DQ / competition review indicators by source procedure subtype')
ax.set_ylabel('%; bidder rates exclude records without bidder publication')
plt.tight_layout();

## 9. Monthly coverage — is any year partial?

In [ ]:
mc = pd.read_parquet(GOLD / 'monthly_metrics.parquet')
mc = mc[mc.date_known].copy()  # the separate unknown-date row stays in the table
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].bar(mc.release_date, mc.tenders, color='steelblue', width=20)
axes[0].plot(mc.release_date, mc.roll12_median, color='red', label='12-month rolling median')
axes[0].set_ylabel('tenders / month')
axes[0].legend()
axes[0].set_title('Tender volume over time')
axes[1].bar(mc.release_date, mc.value_mn, color='seagreen', width=20)
axes[1].set_ylabel('estimated value (EUR millions)')
axes[1].set_title('Estimated value over time')
plt.tight_layout();

## 10. Findings summary

**Dataset**: this notebook prints live Silver row counts in cell 2. Consult the generated findings report for current totals and input hashes; do not cite a hard-coded notebook count.

**Completeness is uneven**: key tender identifiers are strong, while supplier business-ID coverage is incomplete and bidder publication is missing for some records. Bidder rates exclude missing publication rather than treating it as zero. Supplier identity sensitivities are pragmatic groupings, not verified entity resolution.

**Tested key checks**: the current audit found zero dangling tenderer references and no OCID or tender-ID duplicates. This does not assert perfect referential integrity beyond those checks.

**Date-order and value-ratio exceptions are diagnostics**, not evidence of wrongdoing. They require checking source semantics, lot structure, corrigenda, and award state.

**Supplier identity is coverage-sensitive**: every concentration view must show business-ID-only, name-first and source-party-first sensitivities with row/value coverage. Source party IDs are not verified legal identifiers.

**No national-market conclusion is supported.** HHI over all public purchases or broad categories is descriptive procurement concentration, not a competition-law market statistic.

**Transparent review indicators**:
* single-bidding is reported separately for standard-open, price-quotation, minimum-value, negotiated, and other source subtypes, and separately for awarded records
* price-quotation procedures are excluded from the short-standard-open diagnostic
* ratio, date-order, lot reconciliation, zero-bidder, and unawarded-high-value checks remain visible diagnostics but carry zero composite weight
* all thresholds and weights are heuristic review choices, not calibrated probabilities

**Current pipeline**: `gold/tender_features.parquet` is the immutable feature/rule artifact; the ML stage alone writes scored `gold/tender_summary.parquet` using benchmark-elected exploratory detectors plus an HGB rule-proxy explainer.